## Cell 1 — Install and Imports
Install every required package, fix **all** random seeds to 42, and confirm the GPU device.
This cell must be run first; every subsequent cell depends on the imports and the `DEVICE` constant.

In [ ]:
# ── Install (silent mode for Colab) ─────────────────────────────────
!pip install torch optuna matplotlib seaborn scikit-learn numpy pandas -q

import os, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)   # suppress per-trial noise
warnings.filterwarnings('ignore')

# ── Fix ALL random seeds ─────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True    # reproducible CuDNN ops
torch.backends.cudnn.benchmark     = False   # disable auto-tune

# ── Detect device (must show 'cuda' on Colab T4 / A100) ─────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"CUDA   : {torch.version.cuda}")
else:
    print("WARNING: No CUDA GPU detected — training will be slow on CPU.")

## Cell 2 — Dataset Loading and Preprocessing
Load the California Housing dataset (8 numerical features, n=20 640).
Perform an 80 / 20 train-test split (`random_state=42`) and fit a
`StandardScaler` on the training fold **only** to prevent data leakage.

In [ ]:
# Load dataset
housing       = fetch_california_housing()
X             = housing.data           # (20640, 8)
y             = housing.target         # median house value in $100 k
feature_names = list(housing.feature_names)

print(f"Full dataset shape : {X.shape}")
print(f"Feature names      : {feature_names}")
print(f"n_features         : {len(feature_names)}")
print(f"Target range       : [{y.min():.4f}, {y.max():.4f}]")

# ── 80 / 20 train-test split ─────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Train samples : {X_train.shape[0]}")
print(f"Test  samples : {X_test.shape[0]}")

# ── StandardScaler: fit on train only, transform both ────────────────
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)    # fit + transform
X_test_scaled  = scaler.transform(X_test)         # transform only — no leakage

print(f"Scaler mean  : {np.round(scaler.mean_,  4)}")
print(f"Scaler std   : {np.round(scaler.scale_, 4)}")

## Cell 3 — Classical Baselines
Train and evaluate **four** classical regressors on the same 80/20 split:
Linear Regression, Ridge (α=1), Decision Tree (untuned), and Decision Tree
(5-fold GridSearchCV over `max_depth` × `min_samples_split`).
All results are stored in `BASELINE_RESULTS` for later comparison.

In [ ]:
BASELINE_RESULTS = {}

def compute_metrics(model, Xtr, ytr, Xte, yte):
    '''Compute train/test RMSE and R2 for a fitted sklearn estimator.'''
    tr_pred = model.predict(Xtr)
    te_pred = model.predict(Xte)
    tr_rmse = float(np.sqrt(mean_squared_error(ytr, tr_pred)))
    te_rmse = float(np.sqrt(mean_squared_error(yte, te_pred)))
    tr_r2   = float(r2_score(ytr, tr_pred))
    te_r2   = float(r2_score(yte, te_pred))
    return tr_rmse, te_rmse, tr_r2, te_r2

# (a) Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
tr, te, tr2, te2 = compute_metrics(lr_model, X_train_scaled, y_train,
                                    X_test_scaled,  y_test)
BASELINE_RESULTS['Linear Reg'] = dict(train_rmse=tr, test_rmse=te,
                                       train_r2=tr2, test_r2=te2)

# (b) Ridge
rg_model = Ridge(alpha=1.0)
rg_model.fit(X_train_scaled, y_train)
tr, te, tr2, te2 = compute_metrics(rg_model, X_train_scaled, y_train,
                                    X_test_scaled,  y_test)
BASELINE_RESULTS['Ridge'] = dict(train_rmse=tr, test_rmse=te,
                                  train_r2=tr2, test_r2=te2)

# (c) Decision Tree — default, untuned
dt_def = DecisionTreeRegressor(random_state=42)
dt_def.fit(X_train_scaled, y_train)
tr, te, tr2, te2 = compute_metrics(dt_def, X_train_scaled, y_train,
                                    X_test_scaled,  y_test)
BASELINE_RESULTS['DT Default'] = dict(train_rmse=tr, test_rmse=te,
                                       train_r2=tr2, test_r2=te2)

# (d) Decision Tree — GridSearchCV (5-fold, MSE)
param_grid = {
    'max_depth':        [3, 5, 7, 10],
    'min_samples_split': [2, 5, 10]
}
gs = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=5, n_jobs=-1)
gs.fit(X_train_scaled, y_train)
tr, te, tr2, te2 = compute_metrics(gs.best_estimator_,
                                    X_train_scaled, y_train,
                                    X_test_scaled,  y_test)
BASELINE_RESULTS['DT Tuned'] = dict(train_rmse=tr, test_rmse=te,
                                     train_r2=tr2, test_r2=te2,
                                     best_params=gs.best_params_)

print(f"Best DT params : {gs.best_params_}")
print()
W = 72
print('=' * W)
print(f"{'Model':<18} {'Train RMSE':>11} {'Test RMSE':>10} {'Train R2':>9} {'Test R2':>9}")
print('=' * W)
for name, res in BASELINE_RESULTS.items():
    print(f"{name:<18} {res['train_rmse']:>11.4f} {res['test_rmse']:>10.4f} "
          f"{res['train_r2']:>9.4f} {res['test_r2']:>9.4f}")
print('=' * W)

## Cell 4 — Numerical Feature Tokenizer
Each input feature x_i is projected to a `d_token`-dimensional embedding
via a **per-feature** learnable weight W_i and bias b_i:
**token_i = x_i · W_i + b_i**.
This is the feature-tokenisation step of FT-Transformer.

In [ ]:
class NumericalTokenizer(nn.Module):
    '''
    Tokenise every numerical feature into a d_token-dim vector.

    Parameters
    ----------
    n_features : int  number of input features (8 for California Housing)
    d_token    : int  embedding dimension per feature
    '''
    def __init__(self, n_features: int, d_token: int):
        super().__init__()
        # Per-feature weight: shape (F, d_token)
        self.W = nn.Parameter(torch.randn(n_features, d_token))
        # Per-feature bias:   shape (F, d_token)
        self.b = nn.Parameter(torch.zeros(n_features, d_token))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        x      : (B, F)      — flat feature vectors
        return : (B, F, d)   — one token embedding per feature
        '''
        # x.unsqueeze(-1) → (B, F, 1); broadcast-multiply with W (F, d)
        return x.unsqueeze(-1) * self.W + self.b

# ── Unit test ────────────────────────────────────────────────────────
_tok    = NumericalTokenizer(8, 64)
_dummy  = torch.randn(32, 8)
_out    = _tok(_dummy)
status  = 'PASSED' if _out.shape == (32, 8, 64) else 'FAILED'
print(f"NumericalTokenizer unit test : {status}  |  output shape : {_out.shape}")
del _tok, _dummy, _out

## Cell 5 — Transformer Block (Pre-Norm)
A single transformer layer using **Pre-LayerNorm**: LayerNorm is applied
*before* each sub-layer (attention and FFN), with residual connections
wrapping each. Pre-Norm provides more stable gradients than Post-Norm.

In [ ]:
class TransformerBlock(nn.Module):
    '''
    Pre-Norm Transformer block.
    Sub-layers: Multi-Head Self-Attention and Position-wise FFN.
    Each is preceded by LayerNorm and wrapped by a residual connection.
    '''
    def __init__(self, d_token: int, n_heads: int,
                 ffn_factor: float = 1.333, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_token)        # pre-norm for attention
        # Multi-head self-attention (batch_first: B, T, d convention)
        self.attn  = nn.MultiheadAttention(d_token, n_heads,
                                            dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_token)        # pre-norm for FFN
        d_ff = int(d_token * ffn_factor)          # hidden dim of FFN
        self.ffn = nn.Sequential(
            nn.Linear(d_token, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_token),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''x : (B, T, d_token)  — sequence of token embeddings'''
        # Pre-Norm Attention + residual
        normed      = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)   # self-attention
        x = x + attn_out

        # Pre-Norm FFN + residual
        x = x + self.ffn(self.norm2(x))
        return x

# ── Unit test ────────────────────────────────────────────────────────
_blk   = TransformerBlock(64, 4)
_inp   = torch.randn(32, 9, 64)    # batch=32, seq_len=9 (8 features + CLS)
_out   = _blk(_inp)
status = 'PASSED' if _out.shape == (32, 9, 64) else 'FAILED'
print(f"TransformerBlock unit test   : {status}  |  output shape : {_out.shape}")
del _blk, _inp, _out

## Cell 6 — Full FT-Transformer Model
Assemble the complete **Feature Tokenizer + Transformer** architecture:
1. `NumericalTokenizer` maps each feature to a d_token-dim vector.
2. A learnable **[CLS]** token is prepended to the token sequence.
3. N `TransformerBlock` layers process the (F+1) token sequence.
4. The [CLS] output is layer-normalised and projected to a scalar.

In [ ]:
class FTTransformer(nn.Module):
    '''
    Feature Tokenizer + Transformer for tabular regression.

    Architecture
    ------------
    NumericalTokenizer  →  CLS prepend  →  N x TransformerBlock
    →  LayerNorm(CLS)  →  Linear(1)
    '''
    def __init__(self, n_features: int, d_token: int = 64,
                 n_blocks: int = 2, n_heads: int = 4,
                 dropout: float = 0.1):
        super().__init__()
        self.tokenizer = NumericalTokenizer(n_features, d_token)
        # Learnable CLS token — shared parameter, expanded per-batch in forward()
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_token))
        self.blocks    = nn.ModuleList([
            TransformerBlock(d_token, n_heads, dropout=dropout)
            for _ in range(n_blocks)
        ])
        self.norm = nn.LayerNorm(d_token)     # final norm on CLS representation
        self.head = nn.Linear(d_token, 1)     # regression head

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        x      : (B, n_features)  — batch of scaled feature vectors
        return : (B,)             — scalar regression output
        '''
        tokens = self.tokenizer(x)                           # (B, F, d)
        cls    = self.cls_token.expand(x.size(0), -1, -1)   # (B, 1, d)
        tokens = torch.cat([cls, tokens], dim=1)             # (B, F+1, d)
        for block in self.blocks:
            tokens = block(tokens)                           # (B, F+1, d)
        cls_repr = self.norm(tokens[:, 0])                   # (B, d)
        return self.head(cls_repr).squeeze(-1)               # (B,)

# ── Parameter count (default config) ────────────────────────────────
_m    = FTTransformer(n_features=8)   # d_token=64, n_blocks=2, n_heads=4
n_par = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"FTTransformer (default config) — trainable parameters : {n_par:,}")
del _m

## Cell 7 — Dataset and DataLoader Setup
Convert scaled numpy arrays to `float32` tensors, then carve a 90 / 10
inner train / validation split from the training set (used by early stopping
and the LR scheduler). Wrap everything in `DataLoader` objects.

In [ ]:
# Convert to float32 tensors
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_scaled,  dtype=torch.float32)
y_train_t = torch.tensor(y_train,         dtype=torch.float32)
y_test_t  = torch.tensor(y_test,          dtype=torch.float32)

# ── Inner 90 / 10 split (reproducible) ───────────────────────────────
n_total    = len(X_train_t)
n_val      = int(n_total * 0.1)
n_tr_inner = n_total - n_val

gen = torch.Generator()
gen.manual_seed(42)
full_ds = TensorDataset(X_train_t, y_train_t)
train_inner_ds, val_ds = torch.utils.data.random_split(
    full_ds, [n_tr_inner, n_val], generator=gen)

# Extract indices for later evaluate() calls
val_indices   = list(val_ds.indices)
train_indices = list(train_inner_ds.indices)

X_val_t    = X_train_t[val_indices]      # validation features (tensor)
y_val_np   = y_train[val_indices]        # validation targets  (numpy)
y_test_np  = y_test                      # test targets        (numpy)
y_train_np = y_train[train_indices]      # inner-train targets (numpy)

# ── DataLoaders ───────────────────────────────────────────────────────
_pin = DEVICE.type == 'cuda'    # pin_memory only useful with GPU
train_loader = DataLoader(train_inner_ds, batch_size=256,
                          shuffle=True,  num_workers=0, pin_memory=_pin)
val_loader   = DataLoader(val_ds,         batch_size=512,
                          shuffle=False, num_workers=0, pin_memory=_pin)
test_loader  = DataLoader(TensorDataset(X_test_t, y_test_t),
                          batch_size=512, shuffle=False,
                          num_workers=0, pin_memory=_pin)

print(f"Inner-train size : {n_tr_inner}")
print(f"Validation  size : {n_val}")
print(f"Test        size : {len(X_test_t)}")
print(f"Train batches    : {len(train_loader)}")

## Cell 8 — Training Utilities
`train_epoch` runs one forward/backward pass over the training loader
with gradient clipping.  `evaluate` computes RMSE, R², and predictions
in inference mode (no gradients).

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    '''
    One full epoch over the training DataLoader.
    Returns average MSE loss (weighted by sample count).
    '''
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)   # move batch to GPU
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        # Clip gradient norm to 1.0 to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(xb)      # accumulate weighted loss
    return total_loss / len(loader.dataset)      # mean loss over all samples


def evaluate(model, X_tensor, y_numpy, device):
    '''
    Inference-mode evaluation: RMSE, R², and raw predictions.
    Returns (rmse: float, r2: float, preds: np.ndarray)
    '''
    model.eval()
    with torch.no_grad():
        preds = model(X_tensor.to(device)).cpu().numpy()
    rmse   = float(np.sqrt(np.mean((preds - y_numpy) ** 2)))
    ss_res = np.sum((preds - y_numpy) ** 2)
    ss_tot = np.sum((y_numpy - y_numpy.mean()) ** 2)
    r2     = float(1.0 - ss_res / ss_tot)
    return rmse, r2, preds

print("Defined: train_epoch()   evaluate()")

## Cell 9 — Main Training Function
`run_training` builds the model, runs the **AdamW + Cosine Annealing**
training loop with early stopping, restores the best validation checkpoint,
and returns all metrics plus the trained model.
Global DataLoaders / tensors from Cell 7 are used directly.

In [ ]:
def run_training(d_token=64, n_blocks=2, n_heads=4, dropout=0.1,
                 lr=1e-3, weight_decay=1e-4, max_epochs=150,
                 patience=15, seed=42):
    '''
    Full training loop for FTTransformer.

    Uses globals from Cell 7: train_loader, X_val_t, y_val_np,
    X_test_t, y_test_np.

    Returns dict with: model, test_rmse, test_r2, val_rmse,
    best_epoch, train_losses, val_losses, preds.
    '''
    torch.manual_seed(seed)     # per-run reproducibility

    # Build model on the accelerator
    model = FTTransformer(8, d_token, n_blocks, n_heads, dropout).to(DEVICE)

    # AdamW couples weight decay to all non-bias parameters
    optimizer = torch.optim.AdamW(model.parameters(),
                                   lr=lr, weight_decay=weight_decay)
    # Cosine schedule: LR decays from lr → ~0 over max_epochs
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=max_epochs)
    criterion     = nn.MSELoss()

    best_val_rmse = float('inf')
    best_epoch    = 0
    best_state    = None
    train_losses  = []    # MSE loss per epoch
    val_losses    = []    # validation RMSE per epoch

    for epoch in range(max_epochs):
        train_loss = train_epoch(model, train_loader,
                                  optimizer, criterion, DEVICE)
        val_rmse, _, _ = evaluate(model, X_val_t, y_val_np, DEVICE)
        scheduler.step()

        train_losses.append(train_loss)
        val_losses.append(val_rmse)

        # Checkpoint: deep-copy state dict to decouple from ongoing training
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch    = epoch
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}

        # Early stopping
        if epoch - best_epoch >= patience:
            print(f"  Early stop at epoch {epoch + 1} "
                  f"(best epoch {best_epoch + 1})")
            break

    # Restore best weights and evaluate on held-out test set
    model.load_state_dict(best_state)
    test_rmse, test_r2, preds = evaluate(model, X_test_t, y_test_np, DEVICE)

    return {
        'model':        model,
        'test_rmse':    test_rmse,
        'test_r2':      test_r2,
        'val_rmse':     best_val_rmse,
        'best_epoch':   best_epoch,
        'train_losses': train_losses,
        'val_losses':   val_losses,
        'preds':        preds,
    }

print("Defined: run_training()")

## Cell 10 — Default Training Run
Train the FT-Transformer with the default configuration
(d_token=64, n_blocks=2, n_heads=4) and compare with all four classical baselines.
Also defines `train_inner_X` / `train_inner_y` used by subsequent ablation cells.

In [ ]:
print("Training FT-Transformer (default config) ...")
t0 = time.time()

default_result = run_training(d_token=64, n_blocks=2, n_heads=4,
                              dropout=0.1, lr=1e-3, weight_decay=1e-4,
                              max_epochs=150, patience=15, seed=42)
elapsed = time.time() - t0

# Inner-train tensors shared by ablation cells (Cells 11, 12, 17, 19)
train_inner_X = X_train_t[train_indices]    # (n_tr_inner, 8)
train_inner_y = y_train_np                  # numpy inner-train targets

# Compute train RMSE on the inner-train split with the restored model
tr_rmse_default, tr_r2_default, _ = evaluate(
    default_result['model'], train_inner_X, train_inner_y, DEVICE)

print(f"Training time  : {elapsed:.1f} s")
print(f"Best epoch     : {default_result['best_epoch'] + 1}")
print(f"Val  RMSE      : {default_result['val_rmse']:.4f}")
print(f"Test RMSE      : {default_result['test_rmse']:.4f}")
print(f"Test R2        : {default_result['test_r2']:.4f}")

# ── Comparison table ─────────────────────────────────────────────────
W = 72
print()
print('=' * W)
print(f"{'Model':<18} {'Train RMSE':>11} {'Test RMSE':>10} {'Train R2':>9} {'Test R2':>9}")
print('=' * W)
for name, res in BASELINE_RESULTS.items():
    print(f"{name:<18} {res['train_rmse']:>11.4f} {res['test_rmse']:>10.4f} "
          f"{res['train_r2']:>9.4f} {res['test_r2']:>9.4f}")
print(f"{'FT-Transformer':<18} {tr_rmse_default:>11.4f} "
      f"{default_result['test_rmse']:>10.4f} "
      f"{tr_r2_default:>9.4f} {default_result['test_r2']:>9.4f}")
print('=' * W)

## Cell 11 — Ablation: Number of Attention Heads
Vary `n_heads` ∈ {1, 4, 8} while keeping `d_token=64` and `n_blocks=2` fixed.
Results stored in `ABLATION_HEADS` for the heatmap in Cell 17.

In [ ]:
ABLATION_HEADS = {}
print("Ablation — attention heads  (d_token=64, n_blocks=2)")
print('=' * 58)
print(f"{'n_heads':>8} {'Train RMSE':>12} {'Test RMSE':>11} {'Test R2':>9}")
print('=' * 58)

for n_h in [1, 4, 8]:
    res = run_training(d_token=64, n_blocks=2, n_heads=n_h,
                       max_epochs=150, patience=15, seed=42)
    # Compute train RMSE using the same inner-train split
    tr_rmse, _, _ = evaluate(res['model'], train_inner_X, train_inner_y, DEVICE)
    ABLATION_HEADS[n_h] = dict(
        train_rmse = tr_rmse,
        test_rmse  = res['test_rmse'],
        test_r2    = res['test_r2'],
    )
    print(f"{n_h:>8} {tr_rmse:>12.4f} {res['test_rmse']:>11.4f} "
          f"{res['test_r2']:>9.4f}")

print('=' * 58)

## Cell 12 — Ablation: Transformer Depth
Set `best_heads` = argmin test RMSE from Cell 11, then vary
`n_blocks` ∈ {1, 2, 3}.  Results stored in `ABLATION_DEPTH`.

In [ ]:
# Identify best n_heads from the heads ablation
best_heads = min(ABLATION_HEADS, key=lambda k: ABLATION_HEADS[k]['test_rmse'])
print(f"Best n_heads from ablation : {best_heads}")

ABLATION_DEPTH = {}
print(f"Ablation — transformer depth  (d_token=64, n_heads={best_heads})")
print('=' * 58)
print(f"{'n_blocks':>8} {'Train RMSE':>12} {'Test RMSE':>11} {'Test R2':>9}")
print('=' * 58)

for n_b in [1, 2, 3]:
    res = run_training(d_token=64, n_blocks=n_b, n_heads=best_heads,
                       max_epochs=150, patience=15, seed=42)
    tr_rmse, _, _ = evaluate(res['model'], train_inner_X, train_inner_y, DEVICE)
    ABLATION_DEPTH[n_b] = dict(
        train_rmse = tr_rmse,
        test_rmse  = res['test_rmse'],
        test_r2    = res['test_r2'],
    )
    print(f"{n_b:>8} {tr_rmse:>12.4f} {res['test_rmse']:>11.4f} "
          f"{res['test_r2']:>9.4f}")

print('=' * 58)

## Cell 13 — Optuna Hyperparameter Search
50-trial TPE study minimising validation RMSE.
Search space: `d_token`, `n_blocks`, `n_heads`, `dropout`, `lr`, `weight_decay`.
Trials where `d_token % n_heads != 0` are pruned immediately.

In [ ]:
def objective(trial):
    '''
    Optuna objective: minimise validation RMSE.
    Prune invalid head-dimension configurations.
    '''
    d_token  = trial.suggest_categorical('d_token',  [32, 64, 128, 256])
    n_blocks = trial.suggest_int('n_blocks', 1, 4)
    n_heads  = trial.suggest_categorical('n_heads',  [1, 2, 4, 8])
    # Each head must see an integer slice of d_token
    if d_token % n_heads != 0:
        raise optuna.TrialPruned()
    dropout = trial.suggest_float('dropout', 0.0, 0.3)
    lr      = trial.suggest_float('lr',  1e-4, 1e-2, log=True)
    wd      = trial.suggest_float('wd',  1e-6, 1e-3, log=True)
    result  = run_training(d_token, n_blocks, n_heads, dropout,
                           lr, wd, max_epochs=50, patience=10)
    return result['val_rmse']

# Reproducible TPE sampler
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"Best params   : {study.best_params}")
print(f"Best val RMSE : {study.best_value:.4f}")

## Cell 14 — Final Training with Best Optuna Config
Retrain the Optuna-best configuration three times (seeds 0, 42, 123)
to quantify result variance. Save the best model checkpoint as `model.pt`.

In [ ]:
bp = study.best_params
print(f"Best config : {bp}")
print(f"Training 3 runs with seeds [0, 42, 123] ...")
print('=' * 62)

final_runs   = []           # full result dicts for all 3 seeds
best_model   = None
best_te_rmse = float('inf')

for seed in [0, 42, 123]:
    res = run_training(
        d_token      = bp['d_token'],
        n_blocks     = bp['n_blocks'],
        n_heads      = bp['n_heads'],
        dropout      = bp['dropout'],
        lr           = bp['lr'],
        weight_decay = bp['wd'],
        max_epochs   = 150,
        patience     = 15,
        seed         = seed,
    )
    tr_rmse, _, _ = evaluate(res['model'], train_inner_X, train_inner_y, DEVICE)
    res['train_rmse'] = tr_rmse
    final_runs.append(res)
    print(f"  seed={seed:>3}: train RMSE={tr_rmse:.4f}  "
          f"test RMSE={res['test_rmse']:.4f}  test R2={res['test_r2']:.4f}")
    if res['test_rmse'] < best_te_rmse:
        best_te_rmse = res['test_rmse']
        best_model   = res['model']

rmse_vals = [r['test_rmse'] for r in final_runs]
r2_vals   = [r['test_r2']   for r in final_runs]
print('=' * 62)
print(f"Final Test RMSE : {np.mean(rmse_vals):.4f}  +/-  {np.std(rmse_vals):.4f}")
print(f"Final Test R2   : {np.mean(r2_vals):.4f}  +/-  {np.std(r2_vals):.4f}")

# Train RMSE for the single best model (lowest test RMSE across 3 seeds)
best_train_rmse_final, _, _ = evaluate(best_model, train_inner_X, train_inner_y, DEVICE)

# Save best model weights
torch.save(best_model.state_dict(), 'model.pt')
print("Model saved as model.pt")

## Cell 15 — Figure 1: RMSE Bar Chart
Grouped bar chart comparing Train RMSE and Test RMSE for every model.
Classical baselines use orange tones; FT-Transformer bars are blue.
Value labels are printed on top of each bar.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

model_names = list(BASELINE_RESULTS.keys()) + ['FT-Transformer']
train_rmses = (
    [BASELINE_RESULTS[m]['train_rmse'] for m in BASELINE_RESULTS]
    + [tr_rmse_default]
)
test_rmses = (
    [BASELINE_RESULTS[m]['test_rmse'] for m in BASELINE_RESULTS]
    + [default_result['test_rmse']]
)

n_models   = len(model_names)
n_classical = n_models - 1
x = np.arange(n_models)
w = 0.36

# Colour scheme: orange tones for classicals, blue tones for FT-T
train_colors = ['#FFCC80'] * n_classical + ['#64B5F6']
test_colors  = ['#F57C00'] * n_classical + ['#1565C0']

bars_tr = ax.bar(x - w/2, train_rmses, w, color=train_colors,
                  edgecolor='k', linewidth=0.6, label='Train RMSE')
bars_te = ax.bar(x + w/2, test_rmses,  w, color=test_colors,
                  edgecolor='k', linewidth=0.6, label='Test RMSE')

# Value labels on top of every bar
for bar in list(bars_tr) + list(bars_te):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2.0, h + 0.006,
            f'{h:.3f}', ha='center', va='bottom',
            fontsize=8.5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=18, ha='right', fontsize=11)
ax.set_ylabel('RMSE', fontsize=12)
ax.set_title('Train vs Test RMSE — Classical Baselines vs FT-Transformer',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, max(max(train_rmses), max(test_rmses)) * 1.25)
ax.grid(axis='y', alpha=0.3)
sns.despine(fig)
plt.tight_layout()
plt.savefig('figure1_rmse_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: figure1_rmse_comparison.png")

## Cell 16 — Figure 2: Loss Curves
Training MSE loss and validation RMSE per epoch from the default training run.
A vertical dashed green line marks the best (early-stopping) epoch.

In [ ]:
tr_losses  = default_result['train_losses']
va_losses  = default_result['val_losses']
best_ep    = default_result['best_epoch']
epochs_arr = range(1, len(tr_losses) + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs_arr, tr_losses, label='Train MSE Loss',
        color='steelblue', linewidth=1.8)
ax.plot(epochs_arr, va_losses, label='Val RMSE',
        color='darkorange', linewidth=1.8)
ax.axvline(x=best_ep + 1, color='green', linestyle='--', linewidth=1.5,
           label=f'Best epoch ({best_ep + 1})')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss / RMSE', fontsize=12)
ax.set_title('FT-Transformer Training and Validation Loss Curves',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
sns.despine(fig)
plt.tight_layout()
plt.savefig('figure2_loss_curves.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: figure2_loss_curves.png")

## Cell 17 — Figure 3: Ablation Heatmap
Seaborn `heatmap` of test RMSE across all (n_heads × n_blocks) combinations.
Cached results from Cells 11 & 12 are reused; the remaining cells are computed here.
Lower RMSE = cooler colour (coolwarm_r).

In [ ]:
heads_list  = [1, 4, 8]
blocks_list = [1, 2, 3]

# Build the full 3x3 grid — reuse cached results where possible
heatmap_data = {}
for h in heads_list:
    for b in blocks_list:
        if b == 2 and h in ABLATION_HEADS:
            # From Cell 11 ablation
            heatmap_data[(h, b)] = ABLATION_HEADS[h]['test_rmse']
        elif h == best_heads and b in ABLATION_DEPTH:
            # From Cell 12 ablation
            heatmap_data[(h, b)] = ABLATION_DEPTH[b]['test_rmse']
        else:
            # Run the missing (h, b) combination
            print(f"  Computing n_heads={h}, n_blocks={b} ...")
            res = run_training(d_token=64, n_blocks=b, n_heads=h,
                               max_epochs=100, patience=10, seed=42)
            heatmap_data[(h, b)] = res['test_rmse']

# Assemble 2-D array (rows=heads, cols=blocks)
mat = np.array([[heatmap_data[(h, b)] for b in blocks_list]
                 for h in heads_list])

df_heat = pd.DataFrame(
    mat,
    index   = [f'heads={h}'  for h in heads_list],
    columns = [f'blocks={b}' for b in blocks_list])

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(df_heat, annot=True, fmt='.4f', cmap='coolwarm_r',
            ax=ax, linewidths=0.5, annot_kws={'size': 12})
ax.set_title('Ablation Study — Test RMSE by Attention Heads and Depth',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Number of Transformer Blocks', fontsize=11)
ax.set_ylabel('Number of Attention Heads',    fontsize=11)
plt.tight_layout()
plt.savefig('figure3_ablation_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: figure3_ablation_heatmap.png")

## Cell 18 — Figure 4: Predicted vs Actual
Scatter plot of the FT-Transformer's test-set predictions against true
house values. The **red dashed** y=x line marks perfect predictions.
An inset text box reports RMSE and R².

In [ ]:
actual = y_test_np
preds  = default_result['preds']

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(actual, preds, alpha=0.3, s=9, color='steelblue',
           label='Predictions')

# Perfect-prediction reference line (y = x)
lo = min(float(actual.min()), float(preds.min()))
hi = max(float(actual.max()), float(preds.max()))
ax.plot([lo, hi], [lo, hi], 'r--', linewidth=1.8, label='y = x (perfect)')

# Annotate with metrics
rmse_v  = default_result['test_rmse']
r2_v    = default_result['test_r2']
textstr = "RMSE = {:.4f}\nR\u00b2   = {:.4f}".format(rmse_v, r2_v)
props   = dict(boxstyle='round', facecolor='wheat', alpha=0.65)
ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=12,
        verticalalignment='top', bbox=props)

ax.set_xlabel('Actual House Value', fontsize=12)
ax.set_ylabel('Predicted House Value', fontsize=12)
ax.set_title('FT-Transformer: Predicted vs Actual House Values (Test Set)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
sns.despine(fig)
plt.tight_layout()
plt.savefig('figure4_pred_vs_actual.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: figure4_pred_vs_actual.png")

## Cell 19 — Results Summary
One comprehensive table of **all** results — classical baselines,
ablation configurations, and the final Optuna-tuned FT-Transformer —
formatted for direct copy-paste into a research paper.
Final confirmation message printed at the end.

In [ ]:
W = 98
sep = '=' * W
print(sep)
print(f"{'Model':<32} {'Train RMSE':>11} {'Test RMSE':>11} {'Test R2':>9}  Notes")
print(sep)

# ── Classical baselines ───────────────────────────────────────────────
for name, res in BASELINE_RESULTS.items():
    notes = ''
    if name == 'DT Tuned':
        notes = str(res.get('best_params', ''))
    print(f"{name:<32} {res['train_rmse']:>11.4f} {res['test_rmse']:>11.4f} "
          f"{res['test_r2']:>9.4f}  {notes}")

# ── FT-Transformer default ────────────────────────────────────────────
print(f"{'FT-T default':<32} {tr_rmse_default:>11.4f} "
      f"{default_result['test_rmse']:>11.4f} "
      f"{default_result['test_r2']:>9.4f}  d=64 blocks=2 heads=4")

# ── Ablation: attention heads ─────────────────────────────────────────
for h, res in sorted(ABLATION_HEADS.items()):
    label = f'FT-T heads={h}'
    print(f"{label:<32} {res['train_rmse']:>11.4f} {res['test_rmse']:>11.4f} "
          f"{res['test_r2']:>9.4f}  n_blocks=2  d_token=64")

# ── Ablation: transformer depth ───────────────────────────────────────
for b, res in sorted(ABLATION_DEPTH.items()):
    label = f'FT-T blocks={b}'
    print(f"{label:<32} {res['train_rmse']:>11.4f} {res['test_rmse']:>11.4f} "
          f"{res['test_r2']:>9.4f}  n_heads={best_heads}  d_token=64")

# ── Final Optuna best (mean +/- std over 3 seeds) ────────────────────
rmse_mean = np.mean(rmse_vals)
rmse_std  = np.std(rmse_vals)
r2_mean   = np.mean(r2_vals)
r2_std    = np.std(r2_vals)
bp        = study.best_params
notes_f   = (f"d={bp['d_token']} b={bp['n_blocks']} h={bp['n_heads']} "
             f"+/-{rmse_std:.4f} RMSE")
print(f"{'FT-T Optuna best (mean)':<32} {best_train_rmse_final:>11.4f} "
      f"{rmse_mean:>11.4f} {r2_mean:>9.4f}  {notes_f}")
print(sep)
print(f"FT-T Optuna best  RMSE : {rmse_mean:.4f}  +/-  {rmse_std:.4f}")
print(f"FT-T Optuna best  R2   : {r2_mean:.4f}  +/-  {r2_std:.4f}")
print()
print("All experiments complete. Model saved as model.pt")